Aim

To store daily rainfall readings in a Python list and use list slicing to calculate weekly rainfall totals, identify the wettest and driest weeks, calculate a 7-day moving average and identify dry days.

The experiment also compares the performance of a naive moving average and a running-sum approach using different input sizes.

## Prediction Before Implementation

I predict that Python lists and slicing will make it easy to organise and analyse daily rainfall readings. Weekly totals can be calculated by processing groups of seven readings.

I expect the running-sum moving average to be faster than the naive method for larger datasets because it avoids repeatedly adding the same readings.

I also expect the program to handle empty lists and lists containing fewer than seven readings without producing errors.

## Pseudocode – 7-Day Moving Average

1. Start with the list of daily rainfall readings.
2. Create an empty list to store moving averages.
3. Check whether at least 7 readings are available.
4. Start from the first day.
5. Select 7 consecutive readings.
6. Add the 7 readings and divide the total by 7.
7. Store the calculated average.
8. Move forward by one day.
9. Repeat until all possible 7-day groups are processed.
10. Display the moving averages.

In [1]:

import random

# Generate rainfall readings for 365 days
random.seed(10)
readings = [random.randint(0, 50) for i in range(365)]

# 1. Calculate weekly rainfall totals using slicing
weekly_totals = []

for i in range(0, len(readings), 7):
    week = readings[i:i + 7]
    weekly_totals.append(sum(week))

print("TOTAL NUMBER OF DAYS:", len(readings))
print("\nWEEKLY RAINFALL TOTALS:")
print(weekly_totals)

# 2. Find maximum and minimum rainfall with positions
max_rainfall = max(readings)
min_rainfall = min(readings)

wettest_day = readings.index(max_rainfall) + 1
driest_day = readings.index(min_rainfall) + 1

print("\nMAXIMUM RAINFALL:", max_rainfall, "mm")
print("Wettest day:", wettest_day)

print("\nMINIMUM RAINFALL:", min_rainfall, "mm")
print("Driest day:", driest_day)

# 3. Calculate the 7-day moving average
moving_averages = []

for i in range(len(readings) - 6):
    seven_days = readings[i:i + 7]
    average = sum(seven_days) / 7
    moving_averages.append(round(average, 2))

print("\nNUMBER OF MOVING AVERAGES:", len(moving_averages))
print("First 10 moving averages:", moving_averages[:10])

# 4. Find dry days (0 mm rainfall)
dry_days = []

for i in range(len(readings)):
    if readings[i] == 0:
        dry_days.append(i + 1)

print("\nDRY DAYS:", dry_days)
print("TOTAL DRY DAYS:", len(dry_days))


TOTAL NUMBER OF DAYS: 365

WEEKLY RAINFALL TOTALS:
[144, 163, 142, 162, 203, 191, 98, 202, 202, 119, 170, 144, 161, 178, 145, 183, 162, 142, 171, 218, 186, 186, 178, 97, 140, 194, 233, 198, 210, 138, 192, 138, 163, 157, 274, 220, 173, 173, 145, 221, 202, 199, 204, 234, 234, 142, 201, 138, 175, 160, 234, 167, 23]

MAXIMUM RAINFALL: 50 mm
Wettest day: 185

MINIMUM RAINFALL: 0 mm
Driest day: 6

NUMBER OF MOVING AVERAGES: 359
First 10 moving averages: [20.57, 19.57, 23.71, 22.29, 23.86, 20.14, 20.43, 23.29, 23.57, 22.0]

DRY DAYS: [6, 46, 119, 164, 258]
TOTAL DRY DAYS: 5


## Explanation of List Slicing

1. `readings[7:14]`

   Returns readings from index 7 up to index 13. It contains 7 values, representing the eighth to fourteenth readings.

2. `readings[-7:]`

   Returns the last 7 readings from the list.

3. `readings[::7]`

   Returns every seventh reading, starting from the first reading (index 0).

## Why a List Is Suitable

A list is suitable because rainfall readings are stored in order, can be accessed using indexes, and can be modified when new readings arrive.

## List Operation Costs

- `append()` – O(1) average time; adds an item at the end.
- `insert()` – O(n) time; may shift existing elements.
- `pop()` – O(1) at the end, but O(n) when removing from another position.
- `sort()` – O(n log n) time on average.

## Performance Experiment: Naive vs Running-Sum Moving Average

**Prediction:** The naive method recalculates the sum of every seven-day window. The running-sum method reuses the previous total, so it should be faster for larger datasets.

Both methods should produce identical moving averages.

In [2]:
import time

# Naive moving average
def naive_average(data, window=7):
    averages = []

    for i in range(len(data) - window + 1):
        total = sum(data[i:i + window])
        averages.append(round(total / window, 2))

    return averages


# Running-sum moving average
def running_average(data, window=7):
    averages = []

    if len(data) < window:
        return averages

    current_sum = sum(data[:window])
    averages.append(round(current_sum / window, 2))

    for i in range(window, len(data)):
        current_sum = current_sum + data[i] - data[i - window]
        averages.append(round(current_sum / window, 2))

    return averages


# Test using three dataset sizes
for size in [365, 3650, 36500]:

    data = (readings * (size // len(readings) + 1))[:size]

    start = time.perf_counter()
    naive_result = naive_average(data)
    naive_time = time.perf_counter() - start

    start = time.perf_counter()
    running_result = running_average(data)
    running_time = time.perf_counter() - start

    print("\nDATA SIZE:", size)
    print("Results identical:", naive_result == running_result)
    print("Naive time:", round(naive_time, 6), "seconds")
    print("Running-sum time:", round(running_time, 6), "seconds")


DATA SIZE: 365
Results identical: True
Naive time: 0.000386 seconds
Running-sum time: 0.000253 seconds

DATA SIZE: 3650
Results identical: True
Naive time: 0.003588 seconds
Running-sum time: 0.002855 seconds

DATA SIZE: 36500
Results identical: True
Naive time: 0.039507 seconds
Running-sum time: 0.026506 seconds


## Performance Results and Analysis

The naive and running-sum methods produced identical moving averages for all three dataset sizes.

| Dataset size | Naive time (seconds) | Running-sum time (seconds) |
|---|---:|---:|
| 365 | 0.000386 | 0.000253 |
| 3,650 | 0.003588 | 0.002855 |
| 36,500 | 0.039507 | 0.026506 |

The running-sum method was faster because it updates the previous window total instead of calculating the sum again for every window.

The naive method takes O(n × w) time, where n is the dataset size and w is the window size. The running-sum method takes O(n) time. Both methods store the output averages, requiring O(n) space.

## Final Conclusion

Python lists and slicing make it easy to store, access and analyse daily rainfall records. The experiment also shows that the running-sum technique is more efficient than repeatedly calculating each window total.